In [31]:
import os
import sys
import os
import pandas as pd
from PIL import Image
import glob


# Add project root to system path (for relative imports to work)
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

    
from src import config

In [32]:
root_dir="/data/horse/ws/kein254g-team_project/rtdetr_batch/exp2"
labels_dir=f"{root_dir}/labels"
images_dir="/data/horse/ws/kein254g-team_project/test_flat"
txt_files = glob.glob(f"{labels_dir}/*.txt")
test_image_files = glob.glob(f"{images_dir}/*.jpg")

label_stems = {Path(p).stem for p in txt_files}
unlabeled_images = [p for p in test_image_files if Path(p).stem not in label_stems]



# /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2

rows = []
print(len(txt_files))

def get_image_width_height(image_path, file_name):
    parent_dir = os.path.dirname(labels_dir)
    file_name = os.path.splitext(file_name)[0] + ".jpg"
    #file_name = os.path.splitext(os.path.basename(txt_path))[0]   # tomo_01a877_slice_0148
    image_path = os.path.join(images_dir, f"{file_name}") 

    with Image.open(image_path) as img:
        return img.size  # returns (width, height)  

#for filename in glob.glob(f"{root_dir}/*.jpg"): 
#    im=Image.open(filename)
#    W,H = im.size
#    print(f"Image: {filename}, Width: {W}, Height: {H}")

for file_path in txt_files:
    print(f"\n--- Reading: {file_path} ---")
    with open(file_path, "r", encoding="utf-8") as f:
            file_name = os.path.basename(file_path)            # e.g. "image1.txt"
            file_id = os.path.splitext(file_name)[0]           # e.g. "image1"
            _,id,_,z = file_id.split('_')
            z = int(z)
            cls,confidence,x_center,y_center,width,height = f.readline().strip().split()
            id = "tomo_" + str(id)
            #print(f"cls :{cls}, confidence: {confidence}, x_center: {x_center}, y_center: {y_center}, width: {width}, height: {height}")
            #print(f"file_id: {file_id} , file_name: {file_name}")
            #print(f"id: {id}, z: {z}")

            #tomo_id,Motor axis 0,Motor axis 1,Motor axis 2,Has motor
            #tomo_id - unique identifier of the tomogram. Some tomograms in the train set have multiple motors.
            #Motor axis 0 - the z-coordinate of the motor, i.e., which slice it is located on
            #Motor axis 1 - the y-coordinate of the motor
            #Motor axis 2 - the x-coordinate of the motor

            #print(f"filename: {file_name}, file path: {file_path}, file id: {file_id} ")
            W,H = get_image_width_height(images_dir, file_name)
            print(f"Tomo Id: {id} Image Width: {W}, Height: {H}")

            #denormalize
            x = int(round(float(x_center) * W))
            y = int(round(float(y_center) * H))
            

            rows.append({
                    "tomo_id": id,               # just the name without extension
                    "Motor axis 0": z,            # full .txt file name
                    "Motor axis 1": y,
                    "Motor axis 2": x,
                })
            
for img_path in unlabeled_images:
    file_name = os.path.basename(img_path)           
    file_id = os.path.splitext(file_name)[0]           
    _,id,_,z = file_id.split('_')
   
    id = "tomo_" + str(id)
    print(f"Tomo Id: {id} Image Width: {W}, Height: {H}")

    rows.append({
                    "tomo_id": id,               
                    "Motor axis 0": -1,            
                    "Motor axis 1": -1,
                    "Motor axis 2": -1,
                })
     


df = pd.DataFrame(rows)
df.to_csv("./submission.csv", index=False)
print("CSV created: my_submisson.csv")

    
    

46

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_00e047_slice_0170.txt ---
Tomo Id: tomo_00e047 Image Width: 928, Height: 959

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_01a877_slice_0151.txt ---
Tomo Id: tomo_01a877 Image Width: 928, Height: 960

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_01a877_slice_0139.txt ---
Tomo Id: tomo_01a877 Image Width: 928, Height: 960

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_01a877_slice_0146.txt ---
Tomo Id: tomo_01a877 Image Width: 928, Height: 960

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_00e047_slice_0176.txt ---
Tomo Id: tomo_00e047 Image Width: 928, Height: 959

--- Reading: /data/horse/ws/kein254g-team_project/rtdetr_batch/exp2/labels/tomo_01a877_slice_0153.txt ---
Tomo Id: tomo_01a877 Image Width: 928, Height: 960

--- Reading: /data/horse/ws/kein254g-team_projec